In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    trim,
    when,
    lower,
    split, 
    explode, 
    col, 
    concat, 
    coalesce, 
    broadcast,
    row_number,
    desc
)
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
dbutils.widgets.text('externalfiles_zone', 'None', '07-Externalfiles Zone')
externalfiles_zone = dbutils.widgets.get('externalfiles_zone')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_underlaw_process(
    spark, curated_zone, catalog_landed, reference_zone, externalfiles_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, externalfiles_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.UnderLaw"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "process",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== judicial_underlaw ==================

        judicial_underlaw_final_filters = f"businessdate <= {run_date!r}"

        judicial_underlaw_selected_fields = [
            "CDI",
            "L3869Status",
            "ApplicationStatusDate",
            "LawType",
            "CourtHearingDate",
            "CourtOutcome",
            "ApplicationSubmissionDate",
            "BusinessDate",
            "Portfolio", 
            "VAT", 
        ]

        judicial_underlaw = load_landed_delta_table(
            spark,
            externalfiles_zone,  # type: ignore
            "extjudicial3869",
            judicial_underlaw_final_filters,
            judicial_underlaw_selected_fields,
        )

        # Keep only the latest record per CDI+VAT (max businessdate <= run_date)
         # 2. Explode CDI
        judicial_exploded = judicial_underlaw.withColumn("cdi_value", explode(split(col("CDI"), "/")))
        w = Window.partitionBy("cdi_value", "VAT").orderBy(desc("BusinessDate"))
        judicial_underlaw = judicial_exploded.withColumn("_rn", row_number().over(w)).filter(col("_rn") == 1).drop("_rn")

        # Override BusinessDate with run_date for downstream consistency
        judicial_underlaw = judicial_underlaw.withColumn("BusinessDate", lit(run_date).cast("date"))

        judicial_underlaw = judicial_underlaw.withColumnRenamed(
            "L3869Status", "ApplicationStatus_source"
        )

        # --------------transform judicial_underlaw (judicial_underlaw) data---------------------- 
        
        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = [
            "PortfolioId",
            "PortfolioName",
            "Prefix",
            "unit_code_profits",
        ]

        portfolio = load_landed_portfolio_table(
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        ).filter(col("Prefix").isNotNull())


        # 3. Join με portfolio
        df_joined = judicial_underlaw.join(
            broadcast(portfolio),
            judicial_underlaw["Portfolio"].contains(portfolio["Prefix"]),
            "inner"
        )

        # 4. Τελικό select
        def safe_date(c):
            return when(
                (c.isNull()) | (c == ""),
                lit("1753-01-01").cast("date")
            ).otherwise(c.cast("date"))

        final_judicial_underlaw = df_joined.select(
            col("PortfolioId"),
            lit("3869").alias("Law"),
            concat(col("Prefix"), col("cdi_value")).alias("CDI"),
            judicial_underlaw["ApplicationStatus_source"].alias("ApplicationStatus"),
            safe_date(judicial_underlaw["ApplicationStatusDate"]).alias("ApplicationStatusDate"),
            safe_date(judicial_underlaw["CourtHearingDate"]).alias("CourtDate"),
            judicial_underlaw["CourtOutcome"].alias("CourtOutcome"),
            coalesce(judicial_underlaw["LawType"], lit("N/A")).alias("TypeOfCourt"),
            safe_date(judicial_underlaw["ApplicationSubmissionDate"]).alias("SubmissionDate"),
            safe_date(judicial_underlaw["ApplicationStatusDate"]).alias("RecordDate"),
            judicial_underlaw["VAT"].alias("VAT"),
            judicial_underlaw["BusinessDate"].alias("BusinessDate")
        ).distinct()

    
        # =================  params_lawstatusn3869mapping3rdparty =====================

        lawstatusn3869_extra_filters = None

        if lawstatusn3869_extra_filters is not None:
            lawstatusn3869_extra_filters.append(date_filter)
            lawstatusn3869_final_filters = " AND ".join(lawstatusn3869_extra_filters)
        else:
            lawstatusn3869_final_filters = date_filter

        lawstatusn3869_selected_fields = [
            "LawStatus",
            "ApplicationStatus",
            "BusinessDate",
        ]

        lawstatusn3869 = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "params_lawstatusn3869mapping3rdparty",
            lawstatusn3869_final_filters,
            lawstatusn3869_selected_fields,
        )

        # ================ joins =====================

        judicial_underlaw_portfolio_join_condition = (
            substring(final_judicial_underlaw.CDI, 1, 3) == portfolio["Prefix"]
        )

        lawstatusn3869_judicial_underlaw_join_condition = (
            trim(final_judicial_underlaw.ApplicationStatus)
            == lawstatusn3869.ApplicationStatus
        )

        judicial_underlaw_df = final_judicial_underlaw.join(
            portfolio,
            judicial_underlaw_portfolio_join_condition,
            "left",
        ).join(
            lawstatusn3869,
            lawstatusn3869_judicial_underlaw_join_condition,
            "left",
        ).drop(lawstatusn3869.ApplicationStatus) # dupliated column

        # ================ judicial_underlaw_df ================

        UnderLawId_primary_key_fields = [
            final_judicial_underlaw.CDI,
            final_judicial_underlaw.VAT,
            lit("3869"),
            lit("process"),
        ]

        UnderLawKey_primary_key_fields = [final_judicial_underlaw.CDI, lit("3869")]

        judicial_underlaw_df = (
            judicial_underlaw_df.withColumn(
                "UnderLawId",
                sha2(
                    concat_ws(";", *UnderLawId_primary_key_fields),
                    256,
                ),
            )
            .withColumn("UnderLawKey", concat_ws(";", *UnderLawKey_primary_key_fields))
            .withColumn(
                "InLawFlag",
                when(
                    (
                        lower(trim(final_judicial_underlaw.ApplicationStatus)).isin(
                            "rejected", "cancelled", "resignation","paid off"
                        )
                    ),
                    0,
                ).otherwise(1),
            )
            .withColumn(
                "CalcLawStatus",
                when(
                    lower(trim(final_judicial_underlaw.ApplicationStatus)) == "accepted",
                    "Positive Court Decision",
                )
                .when(
                    lower(trim(final_judicial_underlaw.ApplicationStatus)).isin(
                        "pending", "accepted_pending appeal", "rejected_pending appeal"
                    ),
                    "Pending Court Decision",
                )
                .otherwise("Other"),
            )
        )
        
        # Μετά υπολόγισε το ApplicationStatus (resignation -> Resigned)
        judicial_underlaw_df = judicial_underlaw_df.withColumn(
            "ApplicationStatus",
            when(
                lower(final_judicial_underlaw.ApplicationStatus) == "resignation",
                "Resigned",
            ).otherwise(final_judicial_underlaw.ApplicationStatus),
        )


        underlaw_process = judicial_underlaw_df.select(
            judicial_underlaw_df.UnderLawId.alias("UnderLawId"),
            lit("process").alias("SourceSystem"),
            judicial_underlaw_df.unit_code_profits.cast("string").alias("PortfolioKey"),
            judicial_underlaw_df.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            judicial_underlaw_df.UnderLawKey.alias("UnderLawKey"),
            judicial_underlaw_df.CDI.cast("string").alias("CDI"),
            final_judicial_underlaw.VAT.cast("string").alias("TaxRegistrationNumber"),
            lit(None).alias("ApplicationNumber"),
            judicial_underlaw_df.ApplicationStatus.cast("string").alias(
                "ApplicationStatusCode"
            ),
            judicial_underlaw_df.ApplicationStatus.cast("string").alias(
                "ApplicationStatus"
            ),
            judicial_underlaw_df.ApplicationStatusDate.cast("timestamp").alias(
                "ApplicationStatusDate"
            ),
            judicial_underlaw_df.TypeOfCourt.alias("TypeOfCourt"),
            judicial_underlaw_df.CourtDate.cast("date").alias("CourtDate"),
            judicial_underlaw_df.RecordDate.alias("RecordDate"),
            lit(None).alias("InitiationDate"),
            judicial_underlaw_df.CourtOutcome.alias("CourtOutcomeCode"),
            judicial_underlaw_df.CourtOutcome.alias("CourtOutcome"),
            lit(None).alias("CompletionReasonCode"),
            lit(None).alias("CompletionReason"),
            lit(None).alias("LawSystem"),
            lit(None).alias("PlatformName"),
            judicial_underlaw_df.InLawFlag.cast("int").alias("InLawFlag"),
            judicial_underlaw_df.LawStatus.cast("string").alias("LawStatusCode"),
            judicial_underlaw_df.LawStatus.cast("string").alias("LawStatus"),
            lit(None).alias("CalcLawStatusCode"),
            judicial_underlaw_df.CalcLawStatus.alias("CalcLawStatus"),
            judicial_underlaw_df.SubmissionDate.alias("SubmissionDate"),
            lit("N3869").alias("UnderLawType"),
            lit("N3869").alias("UnderLawTypeCode"),
            lit(None).alias("TemporaryCourtDecisionDate"),
            lit(None).alias("DeductionDate"),
            lit(None).alias("CountyCourtCode"),
            lit(None).alias("CountyCourt"),
            lit(None).alias("InterbankStatus3869Code"),
            lit(None).alias("InterbankStatus3869"),
            lit(None).alias("SubmissionDateGroup"),
            lit(None).alias("ResubmissionStatusFlag"),
            lit(None).alias("ResubmissionStatusCode"),
            lit(None).alias("ResubmissionStage"),
            lit(None).alias("AppealCourtDate"),
            lit(None).alias("CepalEligibilityDate"),
            final_judicial_underlaw.BusinessDate.alias("BusinessDate"),
        )

        # ========================== Append ==========================
        underlaw_process.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                externalfiles_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")

In [0]:
load_curated_underlaw_process(spark, curated_zone, catalog_landed, reference_zone, externalfiles_zone, run_date, schema, env_prefix)